In [1]:
import csv # Imports Python's built-in CSV module for reading and writing CSV files
import time # Used for delays (sleep()) between requests.
from pathlib import Path # Provides a modern way to handle file paths.
from urllib.parse import urlencode # Converts a dictionary into URL query parameters.

In [2]:
import requests # External library used to send HTTP requests to websites/APIs

In [3]:
OUTPUT_DIR = Path("corpus_raw") # Creates a path object where XML files are stored
METADATA_FILE = Path("corpus_metadata.csv") # Metadata about all downloaded documents will be saved here

In [4]:
BASE_URL = "https://www.legislation.gov.uk" # Base URL for all API requests.

In [5]:
REQUEST_TIMEOUT = 30 # Maximum wait time (seconds) before a request fails
RETRY_ATTEMPTS = 3 # Try a failed request up to 3 times.
RETRY_BACKOFF = 5 # Wait period between retries.
POLITE_DELAY = 1.5 # Wait 1.5 seconds between requests so i dont  overload the server.

In [7]:
HEADERS = {"User-Agent": "NTU-MSc-Research-RCCB-Index/1.0 (academic research; contact via university email)"}

In [9]:
START_YEAR = 2008
END_YEAR = 2026
# Study Period

In [10]:
KEYWORDS = [
    "climate change",
    "carbon",
    "greenhouse gas",
    "emissions trading",
    "net zero",
    "energy efficiency",
    "renewable",]

In [11]:
DOC_TYPES = ["ukpga", "uksi"]

In [12]:
# Keywords are search terms used to find legislation.
# ukpga =	UK Public General Acts
# uksi = UK Statutory Instruments

In [13]:
def build_search_queries():
    queries = []
    for doc_type in DOC_TYPES:
        for keyword in KEYWORDS:
            queries.append({"doc_type": doc_type, "keyword": keyword})
    return queries

In [14]:
# FUnction loops through all the keywords and the doc types

In [15]:
def fetch_search_csv(doc_type, keyword):
    params = {
        "title": keyword,
        "year": "{}-{}".format(START_YEAR, END_YEAR),
        "results-count": 500,}
    url = "{}/{}/data.csv?{}".format(BASE_URL, doc_type, urlencode(params))

    for attempt in range(1, RETRY_ATTEMPTS + 1):
        try:
            resp = requests.get(url, headers=HEADERS, timeout=REQUEST_TIMEOUT)
            if resp.status_code == 200:
                return resp.text
            print("    [WARN] {} -> HTTP {} (attempt {}/{})".format(url, resp.status_code, attempt, RETRY_ATTEMPTS))
        except requests.RequestException as exc:
            print("    [ERROR] {} -> {} (attempt {}/{})".format(url, exc, attempt, RETRY_ATTEMPTS))
        if attempt < RETRY_ATTEMPTS:
            time.sleep(RETRY_BACKOFF * attempt)
    return None

In [ ]:
# Searches legislation.gov.uk.
# Search only document titles.
# Return up to 500 results.
# Sends HTTP GET request.
# 200 = success.
# Returns CSV data as text.

In [16]:
def parse_search_csv(csv_text, doc_type, keyword):
    entries = []
    reader = csv.DictReader(csv_text.splitlines())
    for row in reader:
        title = (row.get("TITLE") or "").strip()
        doc_id_url = (row.get("ID") or "").strip()
        xml_link = (row.get("LINK (XML)") or "").strip()
        published = (row.get("PUBLISHED") or row.get("CreationDate") or "").strip()
        summary = (row.get("SUMMARY") or "").strip()

        if not xml_link:
            continue  # no usable download link for this row

        # ID looks like: http://www.legislation.gov.uk/id/uksi/2018/1155
        path = doc_id_url.split("/id/")[-1] if "/id/" in doc_id_url else doc_id_url

        entries.append({
            "doc_id": path.replace("/", "_") if path else xml_link.replace("/", "_"),
            "title": title,
            "path": path,
            "xml_link": xml_link,
            "published": published,
            "summary": summary,
            "matched_query": "{}:{}".format(doc_type, keyword),
        })
    return entries

In [17]:
# Converts CSV text into Python dictionaries.
# Reads CSV rows.
# Loop Through Rows, one document at a time
# Gets title
# Gets legislation identifier URL.
# Gets XML download URL.
# Gets publication date.
# Gets document summary.
# Ignore rows without XML files.
# Create Metadata Record

In [18]:
def discover_documents():
    all_docs = {}  # xml_link -> metadata dict (de-dupes overlapping keyword hits)

    for query in build_search_queries():
        doc_type, keyword = query["doc_type"], query["keyword"]
        print("Searching {} for title containing '{}'...".format(doc_type, keyword))

        csv_text = fetch_search_csv(doc_type, keyword)
        time.sleep(POLITE_DELAY)
        if csv_text is None:
            continue

        entries = parse_search_csv(csv_text, doc_type, keyword)
        for e in entries:
            key = e["xml_link"]
            if key not in all_docs:
                all_docs[key] = e
            else:
                existing_query = all_docs[key]["matched_query"]
                if e["matched_query"] not in existing_query:
                    all_docs[key]["matched_query"] = "{};{}".format(existing_query, e["matched_query"])

        print("    {} results (total unique so far: {})".format(len(entries), len(all_docs)))

    return list(all_docs.values())

In [ ]:
# Find all matching legislation.
# all_docs = {} Dictionary used to remove duplicates.
# Runs every keyword × document type combination.
# Downloads search results.
# Converts them into Python dictionaries.


In [19]:
def download_document(doc):
    """Download one document's CLML XML (using the exact link the search
    API gave us) and return its result status."""
    url = doc["xml_link"]
    out_file = OUTPUT_DIR / "{}.xml".format(doc["doc_id"])

    for attempt in range(1, RETRY_ATTEMPTS + 1):
        try:
            resp = requests.get(url, headers=HEADERS, timeout=REQUEST_TIMEOUT)
            if resp.status_code == 200 and resp.content:
                out_file.write_bytes(resp.content)
                print("  [OK] {} ({:,} bytes)".format(doc["doc_id"], len(resp.content)))
                doc.update(status="success", http_code=resp.status_code, file=str(out_file))
                return doc
            print("  [WARN] {}: HTTP {} (attempt {}/{})".format(doc["doc_id"], resp.status_code, attempt, RETRY_ATTEMPTS))
        except requests.RequestException as exc:
            print("  [ERROR] {}: {} (attempt {}/{})".format(doc["doc_id"], exc, attempt, RETRY_ATTEMPTS))
        if attempt < RETRY_ATTEMPTS:
            time.sleep(RETRY_BACKOFF * attempt)

    doc.update(status="FAILED", http_code="", file="")
    return doc

In [20]:
def main():
    OUTPUT_DIR.mkdir(exist_ok=True)

    print("=" * 70)
    print("STEP 1: Discovering documents that match the inclusion rule")
    print("  Types: {} | Years: {}-{}".format(DOC_TYPES, START_YEAR, END_YEAR))
    print("  Keywords: {}".format(KEYWORDS))
    print("=" * 70)
    documents = discover_documents()
    print("\nDiscovered {} unique candidate documents.\n".format(len(documents)))

    print("=" * 70)
    print("STEP 2: Downloading CLML XML for each candidate")
    print("=" * 70)
    results = []
    for doc in documents:
        results.append(download_document(doc))
        time.sleep(POLITE_DELAY)

    with open(METADATA_FILE, "w", newline="", encoding="utf-8") as f:
        fieldnames = ["doc_id", "title", "path", "published", "summary", "matched_query",
                      "xml_link", "status", "http_code", "file"]
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        writer.writeheader()
        writer.writerows(results)

    succeeded = sum(1 for r in results if r["status"] == "success")
    failed = len(results) - succeeded
    print("\nDone. {} succeeded, {} failed out of {} candidates.".format(succeeded, failed, len(results)))
    print("Full audit trail written to {} -- review this manually:".format(METADATA_FILE))
    print("  - Check 'title' for false positives (e.g. unrelated Acts that happen")
    print("    to contain a keyword) and remove rows you judge out of scope,")
    print("    NOTING the exclusion and reason in your dissertation.")
    print("  - Check for false negatives by spot-checking a few known instruments")
    print("    (e.g. the Climate Change Act 2008) appear in the results.")


if __name__ == "__main__":
    main()

STEP 1: Discovering documents that match the inclusion rule
  Types: ['ukpga', 'uksi'] | Years: 2008-2026
  Keywords: ['climate change', 'carbon', 'greenhouse gas', 'emissions trading', 'net zero', 'energy efficiency', 'renewable']
Searching ukpga for title containing 'climate change'...
    2 results (total unique so far: 2)
Searching ukpga for title containing 'carbon'...
    0 results (total unique so far: 2)
Searching ukpga for title containing 'greenhouse gas'...
    0 results (total unique so far: 2)
Searching ukpga for title containing 'emissions trading'...
    1 results (total unique so far: 3)
Searching ukpga for title containing 'net zero'...
    0 results (total unique so far: 3)
Searching ukpga for title containing 'energy efficiency'...
    0 results (total unique so far: 3)
Searching ukpga for title containing 'renewable'...
    0 results (total unique so far: 3)
Searching uksi for title containing 'climate change'...
    82 results (total unique so far: 85)
Searching uk